**Author:** Javier Lendínez  
**License:** GNU Affero General Public License v3.0 ([LICENSE](../../LICENSE))

# 2D Panel Methods for NACA Airfoils (`panel2d`)

In this notebook, I explain the theory behind the new 2D panel-method module `src/vlm/lib/panel2d.py` and verify, cell by cell, that the implementation behaves as the theory says. The module is what the **Airfoil** page of the web app calls (`POST /api/airfoil/panel`) to draw the Cp curve, the chordwise loading, the panel geometry and the flow field.

The theoretical framework follows **Katz & Plotkin, *Low-Speed Aerodynamics: From Wing Theory to Panel Methods*, Chapter 11 "Two-Dimensional Numerical Solutions"** (printed pages 301-341). I use these sections:

| Book section (printed pages) | Content | Where it appears in `panel2d` |
|---|---|---|
| 11.2.1 Constant-strength source method (317-322) | Neumann condition, source panels, cosine spacing | `_source_local`, `_surface_influence(geo, "source")`, `naca4_panel_nodes` |
| 11.2.2 Constant-strength doublet method (323-327) | Doublet panels, wake panel, Kutta condition, tangential velocity (Eqs. 11.37-11.38) | `_doublet_local`, `_solve_doublet`, `_wake_velocity` |
| 11.2.3 Constant-strength vortex method (327-331) | Vortex panels and the trailing-edge condition | vortex field obtained by rotating the source field in `_solve_source` |
| 11.3.2 Constant-strength doublet method, Dirichlet (339-341) | Tangential velocity from the derivative of the doublet strength (Eq. 11.85); used here only for the self term and a comparison | self term of `Vt` in `_solve_doublet` |

**What is from the book and what is not.** The panel formulations and the equations cited as "Eq. (11.xx)" come from the book. Several implementation choices do **not** come from the book and are labeled as such where they appear: the Hess-Smith combination of sources plus a single global vortex, the minimum-norm solve of the doublet system, the sign and finite-difference estimate of the doublet self term, and the streamline integrator.

**Scope.** Two-dimensional, incompressible, inviscid, irrotational, attached flow around a NACA 4-digit airfoil. There is no viscous drag, no separation or stall, and no compressibility. Any "drag" returned by the module is the numerical residual of the pressure integration, not a physical prediction (d'Alembert's paradox: the ideal value is zero).

## Theory and Practical Workflow

In two-dimensional potential flow the perturbation potential satisfies Laplace's equation away from singularities. The airfoil contour is discretized into panels, and their unknown source or doublet/vortex strengths are selected to satisfy a no-penetration boundary condition. For a lifting solution, a Kutta condition determines the circulation consistently with trailing-edge behavior. Surface tangential velocity gives the pressure coefficient $C_p=1-(V_t/U_\infty)^2$; integrating pressure along the contour gives force coefficients.

The notebook compares the code's source-plus-vortex and doublet formulations and explicitly marks choices that are implementation-specific. All cases assume steady incompressible, inviscid, attached flow; the pressure-integrated drag residual is not viscous profile drag. See Katz and Plotkin, *Low-Speed Aerodynamics*, Chapter 11, and the source-specific references already given below.

**Usage.** Run the cells sequentially from a repository directory. The public entry point `solve_airfoil_panels` accepts an airfoil, chord, incidence, speed, panel count, and method; some later cells inspect private helpers solely for verification. The examples cover geometry, influence kernels, boundary conditions, pressure integration, and refinement.

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    current = start
    while current != current.parent:
        if (current / 'src' / 'vlm').is_dir():
            return current
        current = current.parent
    raise RuntimeError('Could not locate the VLMPy repository root (expected src/vlm).')


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_VLM = REPO_ROOT / 'src' / 'vlm'
if str(SRC_VLM) not in sys.path:
    sys.path.insert(0, str(SRC_VLM))

print(f'Repository root: {REPO_ROOT}')
print(f'VLM source path added to sys.path: {SRC_VLM}')

Repository root: c:\Users\javil\Documents\GitHub\VLMPy
VLM source path added to sys.path: c:\Users\javil\Documents\GitHub\VLMPy\src\vlm


## 1. Dependencies and imports

`lib.panel2d` only needs `numpy` and `matplotlib.path` (for the point-in-polygon test used to mask the interior of the airfoil). The plots use `plotly`, as in the rest of the project. A few cells use the module's underscore-prefixed helpers (`_source_local`, `_doublet_local`, `_surface_influence`, `_solve_source`, `_solve_doublet`) because the point of the notebook is to inspect the intermediate objects; they are not a stable public API.

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from lib import panel2d as p

np.set_printoptions(precision=4, suppress=True, linewidth=120)
print('panel2d public entry point:', p.solve_airfoil_panels.__name__)

panel2d public entry point: solve_airfoil_panels


## 2. Problem statement, conventions and boundary condition

Away from the body the flow is potential, $\mathbf{Q}=\nabla\Phi$, with $\Phi=\Phi_\infty+\varphi$ and $\nabla^2\varphi=0$, where $\varphi$ is generated by singularities (sources, doublets, vortices) distributed on the airfoil contour. The free stream is $\mathbf{Q}_\infty=U_\infty(\cos\alpha,\ \sin\alpha)$.

**Conventions used in `panel2d`** (observed in the code):

- Axes: $x$ along the chord with the leading edge at $x=0$, $z$ up. The geometry is built at **zero incidence** and the **free stream is rotated** by $\alpha$ (the angle of attack is positive nose-up, so $w_\infty=U_\infty\sin\alpha>0$).
- Panel nodes run from the trailing edge of the lower surface, round the leading edge, to the trailing edge of the upper surface. With $\mathbf{t}=(dx,dz)/L$ the outward normal is $\mathbf{n}=(-dz,\,dx)/L$.
- Moment coefficient about $c/4$, normalised by $q_\infty c^2$; sign as in the original course scripts.

**Boundary condition (Neumann, book Eq. 11.4).** On a solid surface the normal velocity must vanish:

$$(u,w)\cdot\mathbf{n}+\mathbf{Q}_\infty\cdot\mathbf{n}=0 \quad\text{at every collocation point.}$$

The first term is the velocity induced by the singularities and the second is the known free-stream contribution, moved to the right-hand side. **Pressure coefficient (book Eq. 11.18):**

$$C_p = 1-\left(\frac{Q}{Q_\infty}\right)^2, \qquad Q=\text{tangential velocity on the surface.}$$

Discretizing the contour into $N$ panels and imposing the condition at the $N$ collocation points gives $N$ linear equations $\sum_j a_{ij}\,\sigma_j = \text{RHS}_i$, with $a_{ij}=(u,w)_{ij}\cdot\mathbf{n}_i$ the normal velocity induced at point $i$ by a **unit-strength** panel $j$.

## 3. Discretization: cosine spacing, collocation points, normals

The book (Section 11.2.1) clusters panels near the leading and trailing edges with the **full-cosine** rule (Eq. 11.25), obtained by dividing a semicircle uniformly in the angle $\beta$:

$$x=\frac{c}{2}\,(1-\cos\beta),\qquad \beta\in[0,\pi].$$

Collocation points sit at the **center of each panel**, and the normals point out of the body. The camber line and thickness follow the NACA 4-digit definitions already used by the web app; with the $-0.1036$ coefficient the thickness polynomial vanishes at $x=c$, so the contour closes at the trailing edge (the gap is of the order of $10^{-17}$ in the code).

In [3]:
CHORD = 1.0
nodes_x, nodes_z = p.naca4_panel_nodes('2412', CHORD, 40)
geo = p.PanelGeometry(nodes_x, nodes_z, CHORD)

print('panels:', geo.n, '| nodes:', len(nodes_x))
print('first node (TE lower):', (nodes_x[0], nodes_z[0]), '| last node (TE upper):', (nodes_x[-1], nodes_z[-1]))
print('leading-edge panel index:', geo.i_le)

# Outward normals: stepping along +n leaves the body, stepping along -n enters it.
step = 1e-4 * CHORD  # the trailing-edge region is only ~1e-3 c thick
outside = ~geo.inside(geo.x_mid + step * geo.n_x, geo.z_mid + step * geo.n_z)
inside = geo.inside(geo.x_mid - step * geo.n_x, geo.z_mid - step * geo.n_z)
assert outside.all() and inside.all(), 'normals are not all outward'
print('all', geo.n, 'normals point out of the body')

# Cosine spacing check: panel length should be smallest at LE/TE and largest near mid-chord.
print(f'shortest panel: {geo.L.min():.4f} c (at index {geo.L.argmin()}); longest: {geo.L.max():.4f} c (at index {geo.L.argmax()})')

scale = 0.05 * CHORD
fig = go.Figure()
fig.add_trace(go.Scatter(x=geo.x, y=geo.z, mode='lines+markers', name='nodes', line=dict(color='#555'), fill='toself', fillcolor='rgba(120,120,120,0.2)'))
fig.add_trace(go.Scatter(x=geo.x_mid, y=geo.z_mid, mode='markers', name='collocation points', marker=dict(color='black', size=5)))
nx, nz = [], []
for xm, zm, a, b in zip(geo.x_mid, geo.z_mid, geo.n_x, geo.n_z):
    nx += [xm, xm + scale * a, None]
    nz += [zm, zm + scale * b, None]
fig.add_trace(go.Scatter(x=nx, y=nz, mode='lines', name='normals', line=dict(color='#d62728', width=1)))
fig.update_layout(title='NACA 2412, 40 panels (cosine spacing)', xaxis=dict(title='x / c', scaleanchor='y'), yaxis=dict(title='z / c'), height=380)
fig.show()

panels: 40 | nodes: 41
first node (TE lower): (np.float64(1.0), np.float64(1.6653345369377347e-17)) | last node (TE upper): (np.float64(1.0), np.float64(-1.6653345369377347e-17))
leading-edge panel index: 19
all 40 normals point out of the body
shortest panel: 0.0062 c (at index 0); longest: 0.0785 c (at index 30)


### Panel-Length Distribution

Cosine spacing concentrates nodes near the leading and trailing edges. Plotting normalized panel length against midpoint position makes that resolution visible independently of the geometry plot. The upper and lower surfaces both appear; this is a mesh diagnostic rather than an aerodynamic validation.

In [ ]:
assert np.isfinite(geo.L).all() and np.all(geo.L > 0)
fig_panel_lengths = go.Figure(go.Scatter(
    x=geo.x_mid / CHORD, y=geo.L / CHORD, mode='markers',
    marker=dict(color=np.arange(geo.n), colorscale='Viridis', colorbar=dict(title='Panel index')),
    name='Panel lengths',
))
fig_panel_lengths.update_layout(
    title='Cosine-spaced panel lengths', xaxis_title='Panel midpoint x / c',
    yaxis_title='Panel length / c', template='plotly_white',
)
fig_panel_lengths.show()

## 4. Influence of a constant-strength panel

In the panel frame (panel from $x_1=0$ to $x_2=L$, point at $(x,z)$ in panel coordinates), the book gives closed-form velocities for each constant-strength element. They are rotated back to global axes with the panel angle (Eqs. 11.21-11.23).

**Source of strength $\sigma$ (Eqs. 11.21-11.22):**

$$u_p=\frac{\sigma}{4\pi}\ln\frac{r_1^2}{r_2^2},\qquad w_p=\frac{\sigma}{2\pi}\left[\tan^{-1}\frac{z}{x-x_2}-\tan^{-1}\frac{z}{x-x_1}\right],\qquad r_{1,2}^2=(x-x_{1,2})^2+z^2.$$

**Doublet of strength $\mu$ (Eqs. 11.30-11.31)**, equivalent to two point vortices of opposite sign at the panel ends:

$$u_p\propto\frac{\mu}{2\pi}\left[\frac{z}{r_1^2}-\frac{z}{r_2^2}\right],\qquad w_p\propto\frac{\mu}{2\pi}\left[\frac{x-x_1}{r_1^2}-\frac{x-x_2}{r_2^2}\right].$$

In `panel2d` the sign of the doublet kernel is the one of the original course scripts (it is the negative of the form above; the same convention is used consistently for the matrix, the wake, the field, and the lift formula, and the lift sign is validated against the source method in Section 8).

**Vortex.** A constant vortex panel is the source panel rotated by 90 degrees: $(u,w)_{\text{vortex}}=(w_s,\,-u_s)$ in the panel frame. This is what `_solve_source` uses.

**Self-induced term.** At the center of its own panel, evaluated on the outer side ($z=0^+$), a source gives $w_p=\sigma/2$ and $u_p=0$ (Eq. 11.29). A doublet gives $u_p=0$ and a normal velocity of magnitude $2\mu/(\pi L)$ (Eq. 11.35a). Both are imposed in `_surface_influence` on the matrix diagonal.

### Verification against numerical integration

A panel of constant strength is the integral of point singularities along the panel. I integrate the point-source kernel over $[0,L]$ with the trapezoidal rule and compare it with the closed form, and I check that the doublet is the $z$-derivative of the source (a doublet is a dipole built from a source/sink pair).

In [4]:
L_panel, xi, zi = 0.7, 0.4, 0.15
xs = np.linspace(0.0, L_panel, 200_001)
r2 = (xi - xs) ** 2 + zi ** 2

# Point source of unit strength: u = (x - xs) / (2 pi r^2), w = z / (2 pi r^2).
u_quad = np.trapezoid((xi - xs) / r2, xs) / (2 * np.pi)
w_quad = np.trapezoid(zi / r2, xs) / (2 * np.pi)
u_closed, w_closed = p._source_local(np.array([xi]), np.array([zi]), L_panel)
print(f'source  u: quadrature {u_quad:.10f} | closed form {u_closed[0]:.10f}')
print(f'source  w: quadrature {w_quad:.10f} | closed form {w_closed[0]:.10f}')
assert np.isclose(u_quad, u_closed[0], atol=1e-9) and np.isclose(w_quad, w_closed[0], atol=1e-9)

# Doublet = -d/dz of the source velocity (central finite difference), per the module's sign convention.
h = 1e-6
us_p, ws_p = p._source_local(np.array([xi]), np.array([zi + h]), L_panel)
us_m, ws_m = p._source_local(np.array([xi]), np.array([zi - h]), L_panel)
u_d, w_d = p._doublet_local(np.array([xi]), np.array([zi]), L_panel)
print(f'doublet u: kernel {u_d[0]:.8f} | -d(u_src)/dz {-(us_p - us_m)[0] / (2 * h):.8f}')
print(f'doublet w: kernel {w_d[0]:.8f} | -d(w_src)/dz {-(ws_p - ws_m)[0] / (2 * h):.8f}')
assert np.isclose(u_d[0], -(us_p - us_m)[0] / (2 * h), atol=1e-6) and np.isclose(w_d[0], -(ws_p - ws_m)[0] / (2 * h), atol=1e-6)
print('influence kernels verified')

source  u: quadrature 0.0384993381 | closed form 0.0384993381
source  w: quadrature 0.3691080656 | closed form 0.3691080656
doublet u: kernel 0.08139431 | -d(u_src)/dz 0.08139431
doublet w: kernel 0.77324593 | -d(w_src)/dz 0.77324593
influence kernels verified


## 5. Source method on a non-lifting body (book Section 11.2.1)

With only source panels and the Neumann condition, the system is $N\times N$:

$$\sum_{j=1}^{N} a_{ij}\,\sigma_j=-\mathbf{Q}_\infty\cdot\mathbf{n}_i,\qquad a_{ij}=(u,w)_{ij}\cdot\mathbf{n}_i,\quad a_{ii}=\tfrac12 .$$

The book notes that this method does not use a Kutta condition, so the airfoil circulation is zero and **no lift or drag is produced**, but the pressure distribution of a symmetric body at $\alpha=0$ is well predicted. It also notes that for a closed body the total source strength must vanish, $\sum_j\sigma_jL_j=0$, a useful numerical check.

Here I build the pure-source system on a NACA 0012 at $\alpha=0$ from the module's influence helper and compare with the full solver. For a symmetric airfoil at zero incidence the full solver (Section 6) must return a vortex strength $\gamma\approx0$, so both Cp curves have to coincide.

In [5]:
geo0 = p.PanelGeometry(*p.naca4_panel_nodes('0012', 1.0, 100), 1.0)
u_s, w_s = p._surface_influence(geo0, 'source')       # local-frame influence at the control points
UG, WG = geo0.to_global(u_s, w_s)
A_src = UG * geo0.n_x[:, None] + WG * geo0.n_z[:, None]
rhs = -geo0.n_x                                       # alpha = 0, U = 1: -Q_inf . n
sigma = np.linalg.solve(A_src, rhs)

Vt_pure = geo0.t_x + (UG * geo0.t_x[:, None] + WG * geo0.t_z[:, None]) @ sigma
cp_pure = 1.0 - Vt_pure ** 2

closure = np.sum(sigma * geo0.L)
print(f'sum(sigma*L) = {closure:+.2e}  vs  sum(|sigma|*L) = {np.sum(np.abs(sigma) * geo0.L):.3f}  (closed body: small compared with the right-hand number; it is not exactly 0 at finite N)')
print(f'max |sigma| = {np.abs(sigma).max():.3f}; cond(A) = {np.linalg.cond(A_src):.1f}')

ref = p.solve_airfoil_panels('0012', 1.0, 0.0, 1.0, 100, 'source', with_field=False)
cp_ref = np.array(ref['distribution']['cp'])
print(f"solver gamma = {ref['distribution']['gamma']:+.2e}, CL = {ref['coefficients']['cl']:+.2e}")
print(f'max |Cp_pure_source - Cp_solver| = {np.max(np.abs(cp_pure - cp_ref)):.2e}')
assert np.max(np.abs(cp_pure - cp_ref)) < 1e-5 and abs(closure) < 2e-2 * np.sum(np.abs(sigma) * geo0.L)

fig = go.Figure()
fig.add_trace(go.Scatter(x=geo0.x_mid, y=cp_pure, mode='lines', name='pure sources (this cell)'))
fig.add_trace(go.Scatter(x=geo0.x_mid, y=cp_ref, mode='markers', name='panel2d solver', marker=dict(size=4)))
fig.update_layout(title='NACA 0012, alpha = 0: Cp from sources only', xaxis_title='x / c', yaxis=dict(title='Cp', autorange='reversed'), height=360)
fig.show()

sum(sigma*L) = +1.94e-03  vs  sum(|sigma|*L) = 0.281  (closed body: small compared with the right-hand number; it is not exactly 0 at finite N)
max |sigma| = 1.305; cond(A) = 56.9
solver gamma = -5.15e-18, CL = -2.60e-17
max |Cp_pure_source - Cp_solver| = 4.98e-07


## 6. Lifting flow: sources plus a global vortex (Hess-Smith) with a Kutta condition

To obtain lift, circulation must be added and a **Kutta condition** must select it. The book handles this in two ways: with doublets and a wake (Section 11.2.2) or with vortex panels (Section 11.2.3). The vortex method imposes $\gamma_1+\gamma_N=0$ at the two trailing-edge panels (Eq. 11.52), which, with one equation per panel, leaves $N+1$ equations for $N$ unknowns, so the book removes one equation.

`panel2d` uses the classical **Hess-Smith** construction (an implementation choice, not a section of the book): one unknown source $\sigma_j$ per panel **plus one common vortex strength $\gamma$** on every panel. The unknowns are $\mathbf{x}=[\sigma_1,\dots,\sigma_N,\gamma]$ and the system is square with $N+1$ equations:

$$\begin{pmatrix} A_{\sigma} & \mathbf{a}_\gamma \\ \mathbf{k}_\sigma^T & k_\gamma \end{pmatrix}\begin{pmatrix}\boldsymbol\sigma\\ \gamma\end{pmatrix}=\begin{pmatrix}-\mathbf{Q}_\infty\cdot\mathbf{n}\\ -\mathbf{Q}_\infty\cdot(\mathbf{t}_1+\mathbf{t}_N)\end{pmatrix}$$

- The first $N$ rows are the Neumann condition; $\mathbf{a}_\gamma$ is the normal velocity induced at each collocation point by the vortex distribution of unit $\gamma$ on all panels.
- The last row is the Kutta condition expressed as in the book's vortex method: the tangential velocities at the two trailing-edge panels cancel, $V_{t,1}+V_{t,N}=0$ (the flow leaves the trailing edge smoothly).

**Loads.** The pressure coefficient follows Eq. 11.18 with the total tangential velocity $V_t=\mathbf{Q}_\infty\cdot\mathbf{t}+(\text{induced})\cdot\mathbf{t}$. Forces are obtained by integrating the pressure over the contour,

$$C_x=-\frac{1}{c}\sum_i C_{p,i}\,n_{x,i}\,L_i,\quad C_z=-\frac{1}{c}\sum_i C_{p,i}\,n_{z,i}\,L_i,\quad C_L=-C_x\sin\alpha+C_z\cos\alpha,\quad C_D=C_x\cos\alpha+C_z\sin\alpha,$$

and, as an independent check, from the Kutta-Joukowski theorem with the total circulation $\Gamma=\gamma\sum_jL_j$:

$$C_L=\frac{2\Gamma}{U_\infty c}.$$

### Check against thin-airfoil theory

For a thin cambered airfoil, $C_L\approx 2\pi(\alpha-\alpha_{L0})$ with $\alpha_{L0}=-\frac1\pi\int_0^\pi\frac{dz_c}{dx}(\cos\theta-1)\,d\theta$. This is a first-order **plausibility check**, not a validation: the panel method includes thickness and the exact boundary condition on the real contour, so small differences are expected.

In [6]:
def thin_airfoil_alpha_l0(code):
    m, pos = int(code[0]) / 100.0, int(code[1]) / 10.0
    theta = np.linspace(0.0, np.pi, 20_001)
    x = 0.5 * (1.0 - np.cos(theta))
    if m == 0.0 or pos == 0.0:
        return 0.0
    slope = np.where(x < pos, 2 * m / pos ** 2 * (pos - x), 2 * m / (1 - pos) ** 2 * (pos - x))
    return -np.trapezoid(slope * (np.cos(theta) - 1.0), theta) / np.pi


code_naca, alpha_deg = '2412', 5.0
a_l0 = thin_airfoil_alpha_l0(code_naca)
cl_thin = 2 * np.pi * (np.radians(alpha_deg) - a_l0)

hs = p.solve_airfoil_panels(code_naca, 1.0, alpha_deg, 1.0, 200, 'source', with_field=False)
c, d = hs['coefficients'], hs['diagnostics']
print(f'thin-airfoil alpha_L0 = {np.degrees(a_l0):+.3f} deg -> CL_thin = {cl_thin:.4f}')
print(f"panel CL from pressure integral : {c['cl_pressure']:.4f}")
print(f"panel CL from Kutta-Joukowski   : {c['cl_kutta']:.4f}   (relative difference {abs(c['cl_kutta'] / c['cl_pressure'] - 1) * 100:.2f} %)")
print(f"Cm(c/4) = {c['cm_quarter']:+.4f} | CD residual (ideal 0) = {c['cd_pressure']:+.1e}")
print(f"system size = {hs['n_panels'] + 1} x {hs['n_panels'] + 1} | cond(A) = {d['cond']:.1f} | max |V.n| = {d['bc_residual']:.1e} | Kutta residual Vt1+VtN = {d['kutta_residual']:+.1e}")
assert d['bc_residual'] < 1e-10 and abs(d['kutta_residual']) < 1e-10
assert abs(c['cl_kutta'] / c['cl_pressure'] - 1) < 0.02

thin-airfoil alpha_L0 = -2.077 deg -> CL_thin = 0.7761
panel CL from pressure integral : 0.8536
panel CL from Kutta-Joukowski   : 0.8591   (relative difference 0.65 %)
Cm(c/4) = -0.0617 | CD residual (ideal 0) = -2.5e-04
system size = 201 x 201 | cond(A) = 135.4 | max |V.n| = 1.6e-15 | Kutta residual Vt1+VtN = -4.4e-16


## 7. Constant doublets with a wake (book Section 11.2.2)

The simplest lifting panel code in the book uses one **constant-strength doublet per panel** and a **wake doublet panel** of strength $\mu_w$ that starts at the trailing edge and extends to infinity (Fig. 11.20). A constant doublet panel is equivalent to two point vortices of strength $\pm\mu$ at its ends, so the net vortex left at the trailing edge by the first and last panels is $\mu_1-\mu_N$ (up to sign convention). The **Kutta condition** (Eq. 11.36) requires zero circulation at the trailing edge, which the wake cancels:

$$\mu_1-\mu_N+\mu_w=0\ \Longrightarrow\ \mu_w=\mu_N-\mu_1 .$$

Substituting $\mu_w$ into the Neumann equations removes the extra unknown and leaves an $N\times N$ system: the wake influence $a_{iw}$ (a point-vortex-like velocity from the trailing edge, `_wake_velocity`) is added to the last column and subtracted from the first, exactly as `_solve_doublet` does.

**Lift.** With $\Gamma$ given by the wake strength, $C_L=2\Gamma/(U_\infty c)$. In `panel2d` the wake strength is `gamma = mu[-1] - mu[0]` and $C_L=-2\,\gamma/(U_\infty c)$; the minus sign comes from the sign convention of the doublet kernel (Section 4) and is validated numerically below.

### Observation: the doublet matrix is singular
For a closed contour of **uniform** doublet strength, $\mu_j=\text{const}$, the net velocity outside is zero and the wake term $\mu_N-\mu_1$ vanishes. A uniform $\boldsymbol\mu$ is therefore a null vector of the Neumann matrix: every row of $A$ sums to (numerically) zero and one singular value is at machine precision. This is my own observation of the implementation, not a statement from the book. The consequence is harmless for the quantities used: the loads depend on $\mu_N-\mu_1$ and on differences of $\mu$ between neighbors, which are invariant under adding a constant. `_solve_doublet` therefore uses `numpy.linalg.lstsq` (minimum-norm solution, `rcond=1e-10`) instead of `numpy.linalg.solve`.

In [7]:
geo_d = p.PanelGeometry(*p.naca4_panel_nodes('2412', 1.0, 100), 1.0)
alpha = np.radians(5.0)
sol_d = p._solve_doublet(geo_d, 1.0, alpha)
A_d, mu = sol_d['A'], sol_d['strength']
b_d = -(np.cos(alpha) * geo_d.n_x + np.sin(alpha) * geo_d.n_z)

sv = np.linalg.svd(A_d, compute_uv=False)
print(f'max |row sum of A|      = {np.max(np.abs(A_d.sum(axis=1))):.1e}   (scale of entries: {np.abs(A_d).max():.0f})')
print(f'smallest singular values = {sv[-3:]}')
print(f'cond without the null direction (sv[0]/sv[-2]) = {sv[0] / sv[-2]:.0f}')

# Shift invariance: adding a constant to mu changes neither the residual nor mu_N - mu_1.
shift = 7.3
print(f'||A mu - b||_inf  = {np.max(np.abs(A_d @ mu - b_d)):.2e}')
print(f'||A (mu+c) - b||_inf = {np.max(np.abs(A_d @ (mu + shift) - b_d)):.2e}   (same, as expected)')
print(f"wake strength mu_N - mu_1 = {sol_d['gamma']:+.5f} (identical for mu and mu+c: {(mu + shift)[-1] - (mu + shift)[0]:+.5f})")

res = p.solve_airfoil_panels('2412', 1.0, 5.0, 1.0, 100, 'doublet', with_field=False)
print(f"CL = -2*gamma/(U c) = {res['coefficients']['cl']:.4f}  | CL from pressure integral = {res['coefficients']['cl_pressure']:.4f}")
assert np.max(np.abs(A_d.sum(axis=1))) < 1e-8 and abs(np.max(np.abs(A_d @ (mu + shift) - b_d)) - np.max(np.abs(A_d @ mu - b_d))) < 1e-9
assert res['coefficients']['cl'] > 0

max |row sum of A|      = 4.1e-12   (scale of entries: 322)
smallest singular values = [0.6351 0.3531 0.    ]
cond without the null direction (sv[0]/sv[-2]) = 2415
||A mu - b||_inf  = 1.57e-03
||A (mu+c) - b||_inf = 1.57e-03   (same, as expected)
wake strength mu_N - mu_1 = -0.43184 (identical for mu and mu+c: -0.43184)
CL = -2*gamma/(U c) = 0.8637  | CL from pressure integral = 0.8423


## 8. Pressure from the doublet strength

Once $\mu$ is known, the book gives two routes to the surface tangential velocity: summing the induced velocities of all panels (Eqs. 11.37-11.38, where the self-induced tangential term must be added separately), or, in the Dirichlet formulation of Section 11.3.2, differentiating the potential jump along the surface (Eqs. 11.75-11.76 and 11.85).

`panel2d` follows the first route. At collocation point $i$,

$$V_{t,i}=\mathbf{Q}_\infty\cdot\mathbf{t}_i+\sum_{j}\big[(u,w)_{ij}\cdot\mathbf{t}_i\big]\mu_j+\mu_w\,\big[(u,w)_{iw}\cdot\mathbf{t}_i\big]\;\mp\;\tfrac12\frac{d\mu}{ds}\Big|_i ,$$

where the constant-doublet panel contributes no tangential velocity at its own center, and the last term is the self term of Eq. 11.38 (half the local gradient of the potential jump). In `panel2d` the term is $-\tfrac12\,d\mu/ds$ with $d\mu/ds$ from `numpy.gradient` over the control-point arc length; **the sign follows the doublet-kernel convention of Section 4 and I selected it by comparing with the source solution** (next cell), it is not read from the book. Then $C_p=1-(V_t/U_\infty)^2$.

The cell below checks four variants against the source-method $C_p$ at the same $N$: Eq. 11.37 without the self term, with $+\tfrac12\,d\mu/ds$, with $-\tfrac12\,d\mu/ds$ (the one used), and the "derivative only" estimate $V_t=-d\mu/ds$ (the Dirichlet-style relation of Eq. 11.85, which I used in an earlier version of the module). Away from the two ends (the first and last 10 panels, where all variants degrade) the used variant should be markedly closer to the source solution.

In [8]:
alpha, n_vt = np.radians(5.0), 200
g = p.PanelGeometry(*p.naca4_panel_nodes('2412', 1.0, n_vt), 1.0)
s_sol, d_sol = p._solve_source(g, 1.0, alpha), p._solve_doublet(g, 1.0, alpha)
mu_d = d_sol['strength']

u_i, w_i = p._surface_influence(g, 'doublet')
UG_i, WG_i = g.to_global(u_i, w_i)
uw_i, ww_i = p._wake_velocity(g.x_mid, g.z_mid, g.x[0], g.z[0])
Vt_inf = np.cos(alpha) * g.t_x + np.sin(alpha) * g.t_z
Vt_panels = (UG_i * g.t_x[:, None] + WG_i * g.t_z[:, None]) @ mu_d
Vt_wake = d_sol['gamma'] * (uw_i * g.t_x + ww_i * g.t_z)
arc = np.concatenate(([0.0], np.cumsum(np.hypot(np.diff(g.x_mid), np.diff(g.z_mid)))))
dmu = np.gradient(mu_d, arc, edge_order=2)
base = Vt_inf + Vt_panels + Vt_wake

variants = {
    'Eq. 11.37, no self term': base,
    'self term +1/2 dmu/ds': base + 0.5 * dmu,
    'self term -1/2 dmu/ds (used)': base - 0.5 * dmu,
    'derivative only, -dmu/ds': -dmu,
}
cp_source = 1.0 - s_sol['Vt'] ** 2
err = {}
for name, vt in variants.items():
    diff = np.abs((1.0 - vt ** 2) - cp_source)[10:-10]
    err[name] = np.median(diff)
    print(f'{name:30s} median |Cp - Cp_source| = {np.median(diff):.4f}   max = {diff.max():.3f}')

assert np.allclose(variants['self term -1/2 dmu/ds (used)'], d_sol['Vt']), 'module Vt differs from the variant described here'
assert err['self term -1/2 dmu/ds (used)'] < 0.02 and err['self term -1/2 dmu/ds (used)'] < err['derivative only, -dmu/ds'] / 5

Eq. 11.37, no self term        median |Cp - Cp_source| = 0.6444   max = 2.148
self term +1/2 dmu/ds          median |Cp - Cp_source| = 0.8837   max = 2.834
self term -1/2 dmu/ds (used)   median |Cp - Cp_source| = 0.0118   max = 0.030
derivative only, -dmu/ds       median |Cp - Cp_source| = 0.1180   max = 0.235


Below, I compare the source and doublet Cp curves for the same airfoil and condition. The remaining differences are concentrated in the first and last few panels, where the finite-difference derivative of a piecewise-constant strength is least accurate; the doublet pressure there should be treated with caution.

In [9]:
n_cmp = 200
r_src = p.solve_airfoil_panels('2412', 1.0, 5.0, 1.0, n_cmp, 'source', with_field=False)
r_dbl = p.solve_airfoil_panels('2412', 1.0, 5.0, 1.0, n_cmp, 'doublet', with_field=False)

fig = go.Figure()
for res, label, dash in ((r_src, 'sources + vortex', 'solid'), (r_dbl, 'doublets + wake', 'dash')):
    xm = np.array(res['geometry']['x_mid'])
    cp = np.array(res['distribution']['cp'])
    up, lo = res['distribution']['upper'], res['distribution']['lower']
    fig.add_trace(go.Scatter(x=xm[up], y=cp[up], mode='lines', name=f'{label}: upper', line=dict(dash=dash, color='#d62728')))
    fig.add_trace(go.Scatter(x=xm[lo], y=cp[lo], mode='lines', name=f'{label}: lower', line=dict(dash=dash, color='#1f77b4')))
fig.update_layout(title='NACA 2412, alpha = 5 deg, 200 panels', xaxis_title='x / c', yaxis=dict(title='Cp', autorange='reversed'), height=420)
fig.show()

cp_s, cp_d = np.array(r_src['distribution']['cp']), np.array(r_dbl['distribution']['cp'])
mid = slice(10, -10)
print(f'median |Cp_source - Cp_doublet| away from the ends = {np.median(np.abs(cp_s[mid] - cp_d[mid])):.4f}')
for key in ('cl', 'cl_pressure', 'cm_quarter', 'cp_min'):
    print(f'{key:12s} source {r_src["coefficients"][key]:+.4f} | doublet {r_dbl["coefficients"][key]:+.4f}')

median |Cp_source - Cp_doublet| away from the ends = 0.0118
cl           source +0.8536 | doublet +0.8606
cl_pressure  source +0.8536 | doublet +0.8496
cm_quarter   source -0.0617 | doublet -0.0631
cp_min       source -1.8339 | doublet -1.8044


## 9. Convergence with the number of panels

The book recommends checking convergence by increasing the number of panels (Section 11.1.1). The cell below tabulates $C_L$ (the value the module reports: pressure integral for sources, Kutta-Joukowski for doublets), the $C_L$ from the pressure integral for both, and $C_{m,c/4}$.

In [10]:
panel_counts = (20, 40, 80, 160, 320)
rows = []
for n in panel_counts:
    row = [n]
    for method in ('source', 'doublet'):
        c = p.solve_airfoil_panels('2412', 1.0, 5.0, 1.0, n, method, with_field=False)['coefficients']
        row += [c['cl'], c['cl_pressure'], c['cm_quarter']]
    rows.append(row)
table = np.array(rows)

print(' N  | source: CL     CL(intCp)  Cm      | doublet: CL     CL(intCp)  Cm')
for r in table:
    print(f'{int(r[0]):4d} |        {r[1]:.4f}  {r[2]:.4f}    {r[3]:+.4f} |          {r[4]:.4f}  {r[5]:.4f}    {r[6]:+.4f}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=table[:, 0], y=table[:, 1], mode='lines+markers', name='sources: CL'))
fig.add_trace(go.Scatter(x=table[:, 0], y=table[:, 4], mode='lines+markers', name='doublets: CL (Kutta-Joukowski)'))
fig.add_trace(go.Scatter(x=table[:, 0], y=table[:, 5], mode='lines+markers', name='doublets: CL (pressure integral)', line=dict(dash='dot')))
fig.update_layout(title='Convergence of CL, NACA 2412, alpha = 5 deg', xaxis=dict(title='panels', type='log'), yaxis_title='CL', height=380)
fig.show()

 N  | source: CL     CL(intCp)  Cm      | doublet: CL     CL(intCp)  Cm
  20 |        0.8167  0.8167    -0.0493 |          0.8586  0.7799    -0.0577
  40 |        0.8359  0.8359    -0.0567 |          0.8686  0.8205    -0.0625
  80 |        0.8473  0.8473    -0.0599 |          0.8650  0.8388    -0.0634
 160 |        0.8526  0.8526    -0.0614 |          0.8614  0.8478    -0.0632
 320 |        0.8551  0.8551    -0.0622 |          0.8594  0.8524    -0.0630


### Reading the convergence table
- For sources, the pressure integral and the Kutta-Joukowski value approach each other and a limit as $N$ grows.
- For doublets, the Kutta-Joukowski $C_L$ is almost insensitive to $N$ (it only depends on $\mu_N-\mu_1$), whereas the pressure-integral value converges toward it as $N$ grows because $V_t$ involves a finite difference of $\mu$; in the table the gap between the two shrinks roughly in proportion to $1/N$. At low $N$ the two doublet values differ by several percent. The reported $C_L$ is the Kutta-Joukowski value (what `cl` returns for doublets); the $C_p$ curve and the pressure-integral value should be treated as converged only for large $N$.
- The two formulations are different discretizations of the same continuous problem, so they are not expected to agree to machine precision at finite $N$.

## 10. Velocity field and streamlines

Off the surface, the velocity is the superposition of the free stream and the contribution of every panel (the same panel formulas as in Section 4, now evaluated at arbitrary points):

$$\mathbf{Q}(x,z)=\mathbf{Q}_\infty+\sum_j\mathbf{q}_j(x,z;\sigma_j,\gamma\ \text{or}\ \mu_j)\ \Big[+\ \mu_w\,\mathbf{q}_{\text{wake}}\ \text{for doublets}\Big].$$

For doublets the wake term uses the same strength $\mu_w=\mu_N-\mu_1$ that closed the system (`_wake_velocity`).

`velocity_field` also contains `_safe_zp`, which nudges points with $|z_p|<10^{-9}$ off the panel line to avoid the logarithm/arctangent singularity on a panel.

Streamlines are obtained by integrating $d\mathbf{x}/ds=\mathbf{Q}/|\mathbf{Q}|$ from seeds on the upstream edge of the domain with a midpoint (RK2) step $ds=0.01c$; a streamline stops when it leaves the box or enters the airfoil (point-in-polygon). The color-by-speed and the $C_p$ map are visual aids and add no new physics.

### Near-field warning for constant panels
The book warns that a constant-strength doublet is equivalent to two concentrated vortices at its ends, so **near-field off-surface velocities fluctuate** (Section 11.2.2, and Fig. 11.11 for discrete vortices), even though the velocity at the collocation points and the surface pressure are fine. The same applies, to a lesser degree, to the source panels. The cell below compares the two methods at increasing distance from the upper and lower surface: the difference should drop sharply once the distance exceeds a few panel lengths.

In [11]:
alpha = np.radians(5.0)
print('max |V_source - V_doublet| / U on points offset from the surface (NACA 2412, alpha = 5 deg)')
print('offset/c | 100 panels (above, below) | 200 panels (above, below)')
fields = {}
for n in (100, 200):
    g = p.PanelGeometry(*p.naca4_panel_nodes('2412', 1.0, n), 1.0)
    fields[n] = (g, p._solve_source(g, 1.0, alpha), p._solve_doublet(g, 1.0, alpha))

xs_probe = np.linspace(0.1, 0.9, 200)
diffs = {}
for off in (0.003, 0.01, 0.02, 0.05, 0.1, 0.2):
    row = []
    for n, (g, s_sol, d_sol) in fields.items():
        z_up = np.interp(xs_probe, g.x_mid[g.i_le:], g.z_mid[g.i_le:]) + off
        z_lo = np.interp(xs_probe, g.x_mid[:g.i_le][::-1], g.z_mid[:g.i_le][::-1]) - off
        for z_probe in (z_up, z_lo):
            u1, w1 = p.velocity_field(g, 'source', s_sol, 1.0, alpha, xs_probe, z_probe)
            u2, w2 = p.velocity_field(g, 'doublet', d_sol, 1.0, alpha, xs_probe, z_probe)
            row.append(np.max(np.hypot(u1 - u2, w1 - w2)))
    diffs[off] = row
    print(f'{off:7.3f} | {row[0]:.4f}, {row[1]:.4f}            | {row[2]:.4f}, {row[3]:.4f}')

# The mismatch must shrink with distance and with panel refinement.
assert diffs[0.003][0] > 10 * diffs[0.1][0] and diffs[0.003][2] < diffs[0.003][0]
print('near-field mismatch decays with distance and with panel refinement')

max |V_source - V_doublet| / U on points offset from the surface (NACA 2412, alpha = 5 deg)
offset/c | 100 panels (above, below) | 200 panels (above, below)
  0.003 | 1.5211, 1.0212            | 0.5348, 0.3745
  0.010 | 0.1790, 0.1221            | 0.0285, 0.0215
  0.020 | 0.0328, 0.0259            | 0.0102, 0.0087
  0.050 | 0.0163, 0.0144            | 0.0084, 0.0074
  0.100 | 0.0120, 0.0111            | 0.0062, 0.0057
  0.200 | 0.0073, 0.0073            | 0.0038, 0.0038
near-field mismatch decays with distance and with panel refinement


**Reading the table.** Within about one or two panel lengths of the surface the two fields can differ by order $U_\infty$, so streamlines or $C_p$ maps drawn there, especially for doublets, should not be trusted quantitatively. A few percent away (here $\gtrsim0.05c$ for 100 panels), the two independent formulations agree to about 1-2 percent of $U_\infty$, which is consistent evidence that both fields are correct away from the body. The web app plots the whole domain, so the thin layer next to the surface in the doublet map is a numerical artifact of this near-field behavior, not flow physics.

Finally, the field plots for both methods, as the app draws them (Cp map + streamlines + body), zoomed on the airfoil.

In [12]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=('Sources + vortex', 'Doublets + wake'), horizontal_spacing=0.03)
for col, method in enumerate(('source', 'doublet'), start=1):
    r = p.solve_airfoil_panels('2412', 1.0, 5.0, 1.0, 120, method, with_field=True)
    f = r['field']
    fig.add_trace(go.Heatmap(x=f['x'], y=f['z'], z=f['cp'], zmin=-3, zmax=1, colorscale='RdBu', zsmooth='best', showscale=(col == 2), colorbar=dict(title='Cp')), row=1, col=col)
    for s in f['streamlines']:
        fig.add_trace(go.Scatter(x=s['x'], y=s['z'], mode='lines', line=dict(color='black', width=0.8), hoverinfo='skip', showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=r['geometry']['x'], y=r['geometry']['z'], mode='lines', fill='toself', line=dict(color='#111'), fillcolor='rgba(30,30,36,0.95)', hoverinfo='skip', showlegend=False), row=1, col=col)
    fig.update_xaxes(range=[-0.15, 1.4], scaleanchor=f'y{col if col > 1 else ""}', title_text='x / c', row=1, col=col)
fig.update_yaxes(range=[-0.3, 0.3], title_text='z / c', row=1, col=1)
fig.update_layout(title='Cp map and streamlines, NACA 2412, alpha = 5 deg', height=420)
fig.show()

## 11. Limitations, and what to check before trusting a result

- **Physics.** Inviscid, incompressible, attached flow. No boundary layer, no stall, no compressibility. $C_D$ is only a pressure-integration residual (it should be zero in ideal flow); do not interpret it as drag. For doublets, NACA 2412 at $\alpha=5^\circ$, it was 0.014 at $N=40$ and 0.0003 at $N=400$ in my runs, i.e. it is a discretization error that shrinks with $N$.
- **Geometry.** NACA 4-digit contours only, closed at the trailing edge. The trailing-edge region is very thin, so near-surface probes there need small offsets (see the normal test in Section 3).
- **Wake.** The doublet wake enters only as a point vortex of strength $\mu_N-\mu_1$ at the trailing edge (equivalent to a semi-infinite constant doublet strip, Fig. 11.20); it is not a free wake and does not follow the streamlines. For the streamlines of the app this is an approximation.
- **Singular doublet system.** Solved in the minimum-norm sense; this is justified by the observed null vector (Section 7), but the same reasoning has not been proven for arbitrary contours.
- **Near field.** Do not read velocities within about two panel lengths of the surface (Section 10).
- **Checks that exist in this notebook:** closed-form kernels against numerical integration, the pure-source system against the full solver at $\alpha=0$, Kutta-Joukowski against the pressure integral, the Neumann residual and the Kutta residual, invariance of the doublet solution under a constant shift, and the near-field decay. They verify internal consistency and plausibility against thin-airfoil theory; they are **not** a validation against experimental or published numerical data for the same airfoil.

## 12. References

1. Katz, J. and Plotkin, A., *Low-Speed Aerodynamics: From Wing Theory to Panel Methods*, Chapter 11, "Two-Dimensional Numerical Solutions" (printed pages 301-341): Eq. 11.4 (Neumann condition), Eq. 11.18 ($C_p$), Eqs. 11.21-11.23 (source panel and frame transformation), Eq. 11.25 (cosine spacing), Eq. 11.29 (source self-influence), Eqs. 11.30-11.31 and 11.35a (doublet panel and self-influence), Eq. 11.36 (Kutta condition with wake), Eqs. 11.37-11.38 (tangential velocity), Eqs. 11.44-11.52 (vortex panel and its Kutta condition), Eqs. 11.75-11.76 and 11.85 (tangential velocity from $d\mu/ds$).
2. Hess, J. L. and Smith, A. M. O., "Calculation of potential flow about arbitrary bodies", *Progress in Aerospace Sciences*, 8, 1967 (source of the combined source/global-vortex construction; the construction itself is not given as such in Chapter 11).

## Experimental Context: NACA 0012 Lift-Curve Slope

McCroskey's [NASA TM-100019](https://ntrs.nasa.gov/citations/19880002254) (1987), page 1-4, equation (1), fits the **compressibility-corrected** zero-lift slope $\sqrt{1-M^2}\,dC_l/d\alpha=0.1025+0.00485\log_{10}(Re/10^6)$ per degree for selected Group 1 measurements over $2\times10^6<Re<2\times10^7$, with $M<0.55$. The report gives an RMS fit error of 0.00024 and a maximum fit error of 0.0029 per degree for those points; neither is an uncertainty interval for every experiment.

The comparison below uses $Re=10^7$ and the incompressible limit $M\to0$ of that correlation, with a symmetric NACA 0012 and a central slope over $\alpha=\pm2^\circ$. `panel2d` has **no Reynolds-number, transition, or compressibility input**, so those experimental conditions cannot be matched exactly. The reference is an aggregate correlation, not an individual measured polar or pressure distribution. The discrepancy is reported without an arbitrary acceptance tolerance; this restricted comparison does not validate drag, stall, pressure distributions, or transonic behavior.

In [ ]:
reynolds_reference = 1.0e7
experimental_slope_per_degree = 0.1025 + 0.00485 * np.log10(reynolds_reference / 1.0e6)
naca0012_slope_rows = []
for method in ('source', 'doublet'):
    for requested_panels in (80, 160, 320):
        negative = p.solve_airfoil_panels('0012', 1.0, -2.0, 1.0, requested_panels, method, with_field=False)
        positive = p.solve_airfoil_panels('0012', 1.0, 2.0, 1.0, requested_panels, method, with_field=False)
        assert negative['n_panels'] == positive['n_panels']
        slope_per_degree = (positive['coefficients']['cl'] - negative['coefficients']['cl']) / 4.0
        pressure_slope = (positive['coefficients']['cl_pressure'] - negative['coefficients']['cl_pressure']) / 4.0
        circulation_slope = (positive['coefficients']['cl_kutta'] - negative['coefficients']['cl_kutta']) / 4.0
        assert np.isfinite([slope_per_degree, pressure_slope, circulation_slope]).all()
        naca0012_slope_rows.append({'method': method, 'panels': positive['n_panels'], 'slope_per_degree': slope_per_degree,
                                    'pressure_slope': pressure_slope, 'circulation_slope': circulation_slope})
        difference = (slope_per_degree - experimental_slope_per_degree) / experimental_slope_per_degree
        print(f"{method:7s}, N={positive['n_panels']:3d}: slope={slope_per_degree:.6f} per degree; correlation difference={difference:+.2%}; pressure–circulation={pressure_slope - circulation_slope:+.3e} per degree")
print(f'NASA Group 1 fit at Re=10⁷, incompressible limit: {experimental_slope_per_degree:.6f} per degree')
experimental_figure = go.Figure()
for method in ('source', 'doublet'):
    rows = [row for row in naca0012_slope_rows if row['method'] == method]
    experimental_figure.add_trace(go.Scatter(x=[row['panels'] for row in rows],
        y=[row['slope_per_degree'] for row in rows], mode='lines+markers', name=method))
experimental_figure.add_hline(y=experimental_slope_per_degree, line_dash='dash', annotation_text='NASA correlation, M→0')
experimental_figure.update_layout(title='NACA 0012: lift-slope comparison',
    xaxis_title='Effective panels', yaxis_title='dCl/dα [per degree]', template='plotly_white')
experimental_figure.show()

## Verification and Validation

Kernel quadrature, boundary-condition residuals, pressure/circulation comparisons, and mesh refinements verify mathematical or implementation consistency. They are not experimental validation of viscous drag, stall, or separated flow.

## References

- Katz and Plotkin, *Low-Speed Aerodynamics*, Chapter 11, as specified in the introduction.
- [VLMPy engineering guidance](../README-agent.md) for verification and scope.

## Use Cases

Use this notebook to inspect idealized two-dimensional airfoil pressure distributions, influence kernels, and the behavior of the implemented panel formulations.

## Possible Improvements

Compare against a traceable low-angle experimental pressure dataset with documented airfoil geometry, Reynolds and Mach numbers, transition conditions, and uncertainty. Investigate mesh and near-singularity sensitivity before interpreting differences.

## Next Steps

Compare the distinction between two-dimensional panel results and finite-wing VLM coefficients in [Aerodynamic Coefficients](06_aerodynamic_coefficients.ipynb).